In [ ]:
"""
S01 baseline overflight — integrate notebooks 01–06.

Scenario:
  - One full polar overflight with seeded clouds over the target corridor (notebook 02)
  - 50-target meridian grid (notebook 01)
  - Nadir approach until lead margin before each target, then target engage
  - SequentialTargetBaselinePolicy requests [torque_nm, shutter_gym]; safety applies torque
  - Same training stack as MPO (training_episode_simulation_config; no OBC engage API)
  - Take-picture on every target; memory budget 10 (notebook 06)

Video: torque + pointing telemetry, cumulative latent capture reward, shutter markers.
Verification: s01_utils/baseline_overflight.py
Export: artifacts/07-baseline-overflight.mp4
"""

'\nS01 baseline overflight — integrate notebooks 01–06.\n\nScenario:\n  - One full polar overflight with seeded clouds over the target corridor (notebook 02)\n  - 50-target meridian grid (notebook 01)\n  - Nadir approach until lead margin before each target, then target engage\n  - SequentialTargetBaselinePolicy requests [torque_nm, shutter_gym]; safety applies torque\n  - Same training stack as MPO (training_episode_simulation_config; no OBC engage API)\n  - Take-picture on every target; memory budget 10 (notebook 06)\n\nVideo: torque + pointing telemetry, cumulative latent capture reward, shutter markers.\nVerification: s01_utils/baseline_overflight.py\nExport: artifacts/07-baseline-overflight.mp4\n'

In [ ]:
import os
import sys
from pathlib import Path

notebook_dir = Path.cwd()
backend_root = notebook_dir
for _ in range(6):
    if (backend_root / "simulation").is_dir():
        break
    backend_root = backend_root.parent
os.chdir(backend_root)
sys.path.insert(0, str(backend_root))
_s01_dir = backend_root / "notebooks" / "s01"
sys.path.insert(0, str(_s01_dir))
print(f"backend_root={backend_root}")

backend_root=c:\Users\cedri\code\autonomous-satellite-control-eth-sem-proj\backend


In [ ]:
import importlib

import s01_utils.baseline_overflight as bof

importlib.reload(bof)

setup = bof.build_baseline_overflight_setup()
bof.print_baseline_setup_summary(setup)

Baseline overflight setup
  targets:           50
  clouds:            27 (seeded over target corridor)
  lead margin:       20.0° before target engage
  altitude:          548.2 km
  orbit window:      None° .. None° (auto if unset)
  capture budget:    10/orbit


In [ ]:
rollout = bof.run_baseline_overflight_rollout(setup, show_progress=True)
print(f"steps={rollout.series.t_s.shape[0]}  shutter_cmds={len(rollout.cmd_steps)}")

c:\Users\cedri\code\autonomous-satellite-control-eth-sem-proj\backend\simulation\stepper.py:171: UserWarning: controller_update_interval (1 s) is not an integer multiple of simulation_timestep (0.4 s); using nearest multiple: 0.8 s (2 sim steps).
  self._controller_interval_steps, self._effective_controller_interval_s = resolve_controller_interval_steps(
baseline overflight: 100%|██████████| 2947/2947 [02:44<00:00, 17.90it/s]

steps=2948  shutter_cmds=29


In [ ]:
kpis = bof.evaluate_baseline_capture_results(rollout.series, rollout.cmd_steps)
bof.print_baseline_capture_kpis(kpis, rollout=rollout)

Baseline capture KPIs
  shutter commands:  29 / 50 targets
  captures taken:    10 / 10 budget
  latent capture:    183.96  (k * cov * quality * (1 - cloud))
  applied capture:   183.96  (latent scaled by cov; 0 if not visible / not taken / repeat target)
  mean quality:      0.4093
  budget exhausted at shutter index: 10
  attitude safety events: 0
  mean sim reward:   -88.267

  cmd  cap  taken  visible  cov    quality  cloud   latent  applied  budget_left
  986  986  yes     yes      0.950   0.9255  0.062    82.42    82.42    9
  1006  1006  yes      no      0.000   0.4561  0.000     0.00     0.00    8
  1026  1026  yes      no      0.000   0.3669  1.000     0.00     0.00    7
  1046  1046  yes      no      0.000   0.3448  1.000     0.00     0.00    6
  1066  1066  yes     yes      0.490   0.3383  0.000    16.58    16.58    5
  1086  1086  yes      no      0.000   0.3363  1.000     0.00     0.00    4
  1106  1106  yes     yes      0.540   0.3357  0.000    18.13    18.13    3
  1126 

In [ ]:
import matplotlib

matplotlib.use("Agg")

ARTIFACT_DIR = backend_root / "notebooks" / "s01" / "artifacts"
reward_trace = bof.build_baseline_latent_reward_trace(rollout.series, kpis.capture_results)
video_path = bof.export_baseline_overflight_video(
    rollout.series,
    reward_trace,
    ARTIFACT_DIR / "07-baseline-overflight.mp4",
    shutter_cmd_steps=rollout.cmd_steps,
)

[video] archived previous export -> C:\Users\cedri\code\autonomous-satellite-control-eth-sem-proj\backend\notebooks\s01\artifacts\video_archive\001-07-baseline-overflight.mp4


Writing video: 100%|██████████| 1311/1311 [07:50<00:00,  2.78frame/s]


[mpo_video:after_export] 07-baseline-overflight.mp4 (2928187 bytes, codec=h264)
[mpo_video:play] 07-baseline-overflight.mp4 (2928187 bytes, codec=h264)


artifact=C:\Users\cedri\code\autonomous-satellite-control-eth-sem-proj\backend\notebooks\s01\artifacts\07-baseline-overflight.mp4
